---
title: Creating a SQLite database with parent and child tables
author: Aubrey Moore
date: 2023-02-09
---

# create_one_to_many_sql.ipynb

An example of setting up a SQLite3 database containing a one_to_many relationship between a parent and child tables.

Both tables have a unique id field for each record.

The biggest problem I had was setting up `try ... except` blocks to trap database integrity errors which caused crashes.

In [1]:
import sqlite3
import pandas as pd
from icecream import ic
import os

In [2]:
def create_database(db_path: str):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS people (
            personid INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE,
            age INTEGER
        );
    """)
    conn.commit()
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            noteid INTEGER PRIMARY KEY AUTOINCREMENT,
            personid INTEGER,
            note TEXT,
            FOREIGN KEY(personid) REFERENCES people(personid) ON DELETE CASCADE  
        );
    """)
    conn.commit()
    
    conn.close()
    
# # Example usage:

# create_database('my_database.sqlite3')  

In [3]:
def populate_table_with_dataframe(df: pd.DataFrame, db_path: str, table_name: str) -> pd.DataFrame:
    """
    """
    conn = sqlite3.connect(db_path)
    try:
        df.to_sql(name=table_name, con=conn, if_exists='append', index=False)
        conn.commit()

    except sqlite3.IntegrityError as e:
        # Catch the specific error
        print(f"Caught an integrity error for user ID when populating people table: {e}")
        # Roll back the failed transaction
        conn.rollback()
        print("Transaction rolled back. Continuing with next record.")

    except Exception as e:
        # Catch any other potential errors
        print(f"An unexpected error occurred: {e}")
        print('recovering from error')
        conn.rollback()
    df = pd.read_sql(f'SELECT * FROM {table_name}', con=conn)
    conn.close()
    return df

# MAIN

In [4]:
db_path = 'my_database.sqlite3'
delete_existing_database = True

people_data = [
    {'name': 'Aubrey Moore', 'age': 74},
    {'name': 'Jane Ginlo Moore', 'age': 72}
]

notes_data = [
    {'name': 'Aubrey Moore', 'note': 'an OK guy'},
    {'name': 'Jane Ginlo Moore', 'note': 'a beautiful lady with a beautiful voice'},
    {'name': 'Jane Ginlo Moore', 'note': 'She, who must be obeyed'}
]

In [5]:
if delete_existing_database and os.path.exists(db_path):
    os.remove(db_path)

if not os.path.exists(db_path):
    create_database(db_path)

###########################
# create 'people' dataframe
df = pd.DataFrame(people_data)
ic('before populating db table');
ic(df);

# populate 'people' database table
df = populate_table_with_dataframe(df=df, db_path=db_path, table_name='people')
ic('after populating db table');
ic(df);

# create a dict for name to personid lookup (name -> personid)
personid_map = df.set_index('name')['personid'].to_dict()

##########################
# create 'notes' dataframe reusing the df object)
df = pd.DataFrame(notes_data)
ic('"notes" df before adding "personid"')
ic(df);

# add 'personid' column by using 'name' as a key for the personid_map
df['personid'] = df['name'].map(personid_map)
    
# delete the 'name' column (IMPORTANT)
df.drop('name', inplace=True, axis=1)
ic('"notes" df before populating "notes" db table')
ic(df);

df = populate_table_with_dataframe(df=df, db_path=db_path, table_name='notes')
ic('"notes" df after populating "notes" db table');
ic(df);

ic| 'before populating db table'
ic| df:                name  age
        0      Aubrey Moore   74
        1  Jane Ginlo Moore   72
ic| 'after populating db table'
ic| df:    personid              name  age
        0         1      Aubrey Moore   74
        1         2  Jane Ginlo Moore   72
ic| '"notes" df before adding "personid"'
ic| df:                name                                     note
        0      Aubrey Moore                                an OK guy
        1  Jane Ginlo Moore  a beautiful lady with a beautiful voice
        2  Jane Ginlo Moore                  She, who must be obeyed
ic| '"notes" df before populating "notes" db table'
ic| df:                                       note  personid
        0                                an OK guy         1
        1  a beautiful lady with a beautiful voice         2
        2                  She, who must be obeyed         2
ic| '"notes" df after populating "notes" db table'
ic| df:    noteid  personid               